# Heartbeat Analysis

This notebook inspects the processed heartbeat flow end to end. It checks the shared config, the latest raw and conformed MinIO objects, the curated summary object, and the Trino table when that service is available.

In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from urllib import request

import boto3

CONFIG_PATH = Path("/home/jovyan/config/dags/heartbeat.json")
OUTPUT_PATH = Path("/home/jovyan/data/heartbeat_summary.json")
MINIO_ENDPOINT = "http://minio:9000"
RAW_BUCKET = "raw"
RAW_PREFIX = "reference/heartbeat/events"
CONFORMED_BUCKET = "conformed"
CONFORMED_PREFIX = "reference/heartbeat/events"
CURATED_BUCKET = "curated"
CURATED_KEY = "reference/heartbeat/latest/heartbeat_summary.json"
TRINO_URL = "http://trino:8080/v1/statement"
TRINO_SQL = (
    "SELECT event_id, generated_at, latest_event_timestamp, latest_message, raw_uri, conformed_uri, curated_uri "
    "FROM demo.heartbeat_events ORDER BY latest_event_timestamp DESC LIMIT 20"
)

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Heartbeat config was not found at canonical notebook path {CONFIG_PATH}.")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
config


In [ ]:
def s3_client():
    return boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id="minioadmin",
        aws_secret_access_key="minioadmin",
        region_name="local-01",
    )


def load_json_object(bucket: str, key: str) -> dict[str, object]:
    response = s3_client().get_object(Bucket=bucket, Key=key)
    return json.loads(response["Body"].read().decode("utf-8"))


def find_latest_json_object(bucket: str, prefix: str) -> tuple[str, dict[str, object]]:
    latest_key = None
    for page in s3_client().get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get("Contents", []):
            key = item.get("Key")
            if isinstance(key, str) and key.endswith(".json") and (latest_key is None or key > latest_key):
                latest_key = key
    if latest_key is None:
        raise RuntimeError(f"No JSON objects found under s3://{bucket}/{prefix}.")
    return latest_key, load_json_object(bucket, latest_key)


def run_trino_query(sql: str) -> list[dict[str, object]]:
    headers = {"X-Trino-User": "jovyan", "X-Trino-Catalog": "demo", "X-Trino-Schema": "demo"}
    req = request.Request(TRINO_URL, data=sql.encode("utf-8"), headers=headers, method="POST")
    rows = []
    columns = []

    with request.urlopen(req, timeout=30) as response:
        payload = json.loads(response.read().decode("utf-8"))

    while True:
        if not columns and payload.get("columns"):
            columns = [column["name"] for column in payload["columns"]]
        rows.extend(payload.get("data", []))
        next_uri = payload.get("nextUri")
        if not next_uri:
            break
        with request.urlopen(next_uri, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))

    return [dict(zip(columns, row)) for row in rows]


In [ ]:
generated_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
raw_key, raw_payload = find_latest_json_object(RAW_BUCKET, RAW_PREFIX)
conformed_key, conformed_payload = find_latest_json_object(CONFORMED_BUCKET, CONFORMED_PREFIX)
curated_summary = load_json_object(CURATED_BUCKET, CURATED_KEY)

trino_rows = []
trino_error = None
try:
    trino_rows = run_trino_query(TRINO_SQL)
except Exception as exc:
    trino_error = str(exc)

output_payload = {
    "generated_at": generated_at,
    "config_path": str(CONFIG_PATH),
    "config": config,
    "raw_key": raw_key,
    "raw_payload": raw_payload,
    "conformed_key": conformed_key,
    "conformed_payload": conformed_payload,
    "curated_summary": curated_summary,
    "trino_query": TRINO_SQL,
    "trino_row_count": len(trino_rows),
    "trino_error": trino_error,
    "trino_rows": trino_rows,
}

OUTPUT_PATH.write_text(json.dumps(output_payload, indent=2, sort_keys=True), encoding="utf-8")
output_payload
